In [19]:
import os
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
import json
from datasets import load_dataset


In [20]:
from huggingface_hub import login

hf_token = os.getenv('HF_TOKEN')
if not hf_token:
    print("HuggingFace API key is not working")
else:
    print("HF KEY looks good so far")    
login(hf_token, add_to_git_credential=True)

HF KEY looks good so far


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [21]:
load_dotenv(override=True)
openai_api_key=os.getenv("OPENAI_API_KEY")
if not openai_api_key:
   print("OpenAI API Key not set properly")
else:
    print(f"OpenAI API Key exists")


openai=OpenAI()


OpenAI API Key exists


In [22]:
name_to_symbol = {
    # Apple
    "apple": "Apple Inc.",
    "apple inc": "Apple Inc.",
    "aapl": "Apple Inc.",
    
    # Microsoft
    "microsoft": "Microsoft Corp.",
    "microsoft corp": "Microsoft Corp.",
    "msft": "Microsoft Corp.",
    
    # Tesla
    "tesla": "Tesla Inc.",
    "tsla": "Tesla Inc.",
    
    # Alphabet / Google
    "google": "Alphabet Inc.",
    "alphabet": "Alphabet Inc.",
    "googl": "Alphabet Inc.",
    "googlr": "Alphabet Inc.",  # səhv yazsa belə
    
    # Amazon
    "amazon": "Amazon.com Inc.",
    "amzn": "Amazon.com Inc.",
    
    # JPMorgan
    "jpmorgan": "JPMorgan Chase",
    "jpm": "JPMorgan Chase",
    
    # Visa
    "visa": "VISA Inc.",
    "v": "VISA Inc.",
    
    # Nvidia
    "nvidia": "NVIDIA Corp.",
    "nvda": "NVIDIA Corp.",
    
    # Meta / Facebook
    "meta": "Meta Platforms",
    "facebook": "Meta Platforms",
    
    # UnitedHealth
    "unitedhealth": "UnitedHealth Group",
    "unh": "UnitedHealth Group",
    
    # Home Depot
    "home depot": "Home Depot",
    "hd": "Home Depot",
    
    # Procter & Gamble
    "procter gamble": "Procter & Gamble",
    "pg": "Procter & Gamble",
    
    # Johnson & Johnson
    "johnson johnson": "Johnson & Johnson",
    "jnj": "Johnson & Johnson",
    
    # Mastercard
    "mastercard": "Mastercard Inc.",
    "ma": "Mastercard Inc.",
    
    # Broadcom
    "broadcom": "Broadcom Inc.",
    "avgo": "Broadcom Inc.",
    
    # Walmart
    "walmart": "Wal-Mart Stores",
    "wmt": "Wal-Mart Stores",
    
    # Salesforce
    "salesforce": "Salesforce Inc.",
    "crm": "Salesforce Inc.",
    
    # Adobe
    "adobe": "Adobe Inc.",
    "adbe": "Adobe Inc.",
    
    # Costco
    "costco": "Costco Wholesale",
    "cost": "Costco Wholesale",

    "starbucks":"Starbucks Corp.",
    "nike":"Nike Inc.",
    "walt disney":"Walt Disney Co.",
    "disney":"Walt Disney Co."
}

In [ ]:
import pandas as pd
df = pd.read_csv("hf://datasets/Shoriful025/Stock_Market_Trades/main.csv")

def about_stock(name):
   symbol=name_to_symbol.get(name)
   stock=df[df["symbol"].str.lower()==symbol.lower()]
   if stock.empty:
     print("This stock does not exits in out AI BOT yet.")
   return  stock["price_per_share"],stock["total_volume"],stock["ticker"]
 

about_stock("apple")

(0    185.25
 Name: price_per_share, dtype: float64,
 0    27787.5
 Name: total_volume, dtype: float64,
 0    AAPL
 Name: ticker, dtype: object)

In [24]:
tools=[
    {
        "type": "function",
        "function": {
            "name": "about_stock",
            "description": "Get the full information about stock.",
            "parameters": {
                "type": "object",
                "properties": {
                    "stock_name": {
                        "type": "string",
                        "description": "The name of the stock",
                    }
                },
                "required": ["stock_name"],
                "additionalProperties": False
            }
        }
    }
]

In [31]:
system_prompt="""You are a helpfull assistant for InvestAI which helps people to give bunch of information about 
specific stocks,their price per share,total valume in market and the exchange the stock belongs to.
Please be helpfull and try to give good information to user."""

In [32]:
def handle_tool_call(message):
    responses=[]

    for tool_call in message.tool_calls:
        func_name=tool_call.function.name
        args=json.loads(tool_call.function.arguments)
    
        if func_name=="about_stock":
         stock=args.get("stock_name")
         result=about_stock(stock)

         content=json.dumps(result,ensure_ascii=False) if isinstance(result,dict) else str(result)

         responses.append({
            "role":"tool",
            "content":content,
            "tool_call_id":tool_call.id
         })

    return responses 


In [35]:
def chat_invest(message,history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages=[{"role":"system","content":system_prompt}]+history+[{"role":"user","content":message}]
    response=openai.chat.completions.create(
        model="gpt-5-mini",
        messages=messages,
        tools=tools
    )
    
    while response.choices[0].finish_reason=="tool_calls":
        message=response.choices[0].message
        response=handle_tool_call(message)
        messages.append(message)
        messages.extend(response)
        response=openai.chat.completions.create(model="gpt-5-mini", messages=messages)
       
    return response.choices[0].message.content


In [41]:
gr.ChatInterface(fn=chat_invest,type="messages").launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.
